In [1]:
from google.colab import drive
import os

drive.mount("/content/drive")

RUTA = "/content/drive/MyDrive/radar-micro-Doppler/processed"
print(os.listdir(RUTA))

Mounted at /content/drive
['senales', 'espectrogramas_5s.npz', 'particion.csv', 'metadatos.csv']


In [2]:
import torch
print(torch.__version__)
print("GPU disponible:", torch.cuda.is_available())

2.11.0+cu130
GPU disponible: True


In [4]:
import numpy as np
import pandas as pd

datos = np.load(os.path.join(RUTA, "espectrogramas_5s.npz"))

X = datos["X"]
ids = datos["ids"]

meta = pd.read_csv(os.path.join(RUTA, "metadatos.csv"))
part = pd.read_csv(os.path.join(RUTA, "particion.csv"))

df = meta.merge(part[["id", "particion"]], on="id")

assert (df["id"].values == ids).all(), "El orden de X y del CSV no coincide"

y = df["actividad"].values - 1

mask_train = (df["particion"] == "train").values
mask_valid = (df["particion"] == "valid").values
mask_test  = (df["particion"] == "test").values

X_train, y_train = X[mask_train], y[mask_train]
X_valid, y_valid = X[mask_valid], y[mask_valid]
X_test,  y_test  = X[mask_test],  y[mask_test]

print("train:", X_train.shape, np.bincount(y_train))
print("valid:", X_valid.shape, np.bincount(y_valid))
print("test: ", X_test.shape,  np.bincount(y_test))

train: (1224, 128, 128) [218 218 216 215 215 142]
valid: (253, 128, 128) [46 46 47 47 47 20]
test:  (277, 128, 128) [48 48 48 49 48 36]


In [6]:
import torch
from torch.utils.data import TensorDataset, DataLoader

def a_dataset(X, y):
  X_t = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
  y_t = torch.tensor(y, dtype=torch.long)

  return TensorDataset(X_t, y_t)

ds_train = a_dataset(X_train, y_train)
ds_valid = a_dataset(X_valid, y_valid)
ds_test = a_dataset(X_test, y_test)

BATCH = 32
dl_train = DataLoader(ds_train, batch_size=BATCH, shuffle=True)
dl_valid = DataLoader(ds_valid, batch_size=BATCH, shuffle=False)
dl_test  = DataLoader(ds_test,  batch_size=BATCH, shuffle=False)

imgs, etiquetas = next(iter(dl_train))
print(imgs.shape, imgs.dtype)
print(etiquetas.shape, etiquetas[:10])

torch.Size([32, 1, 128, 128]) torch.float32
torch.Size([32]) tensor([2, 4, 1, 5, 1, 1, 1, 1, 2, 1])


In [11]:
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self, number_of_classes=6):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(16384, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 6)
        )

    def forward(self, x):
        x = self.features(x)
        x = self. classifier(x)

        return x


In [13]:
modelo = CNN()
salida = modelo(imgs)
print(salida.shape)

n_params = sum(p.numel() for p in modelo.parameters())
print(f"Parámetros: {n_params:,}")

torch.Size([32, 6])
Parámetros: 2,121,574


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

modelo = CNN().to(device)
criterio = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)
EPOCAS = 30

In [17]:
import copy

historial = {"loss_train": [], "loss_valid": [], "acc_train": [], "acc_valid": []}
MODELOS = []
validation_error = []

torch.manual_seed(42)
modelo = CNN().to(device)
optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)

for epoca in range(EPOCAS):

    # ---------- FASE DE ENTRENAMIENTO ----------
    modelo.train()
    total_loss, aciertos, n = 0.0, 0, 0

    for imgs, etiquetas in dl_train:
        imgs, etiquetas = imgs.to(device), etiquetas.to(device)

        optimizador.zero_grad()
        salidas = modelo(imgs)
        loss = criterio(salidas, etiquetas)
        loss.backward()
        optimizador.step()

        total_loss += loss.item() * imgs.size(0)
        aciertos += (salidas.argmax(1) == etiquetas).sum().item()
        n += imgs.size(0)

    historial["loss_train"].append(total_loss / n)
    historial["acc_train"].append(aciertos / n)

    # ---------- FASE DE VALIDACIÓN ----------
    modelo.eval()
    total_loss, aciertos, n = 0.0, 0, 0

    with torch.no_grad():
        for imgs, etiquetas in dl_valid:
            imgs, etiquetas = imgs.to(device), etiquetas.to(device)

            salidas = modelo(imgs)
            loss = criterio(salidas, etiquetas)

            total_loss += loss.item() * imgs.size(0)
            aciertos += (salidas.argmax(1) == etiquetas).sum().item()
            n += imgs.size(0)

    validation_error.append(total_loss / n)
    MODELOS.append(copy.deepcopy(modelo.state_dict()))

    historial["loss_valid"].append(total_loss / n)
    historial["acc_valid"].append(aciertos / n)

    print(f"Época {epoca+1:2d} | "
          f"train loss {historial['loss_train'][-1]:.3f} acc {historial['acc_train'][-1]:.3f} | "
          f"valid loss {historial['loss_valid'][-1]:.3f} acc {historial['acc_valid'][-1]:.3f}")

mejor = np.argmin(validation_error)
modelo.load_state_dict(MODELOS[mejor])
print(f"Mejor época: {mejor + 1} (valid loss {validation_error[mejor]:.3f})")

Época  1 | train loss 1.600 acc 0.434 | valid loss 1.441 acc 0.581
Época  2 | train loss 0.966 acc 0.616 | valid loss 0.717 acc 0.814
Época  3 | train loss 0.799 acc 0.676 | valid loss 0.578 acc 0.830
Época  4 | train loss 0.714 acc 0.721 | valid loss 0.522 acc 0.838
Época  5 | train loss 0.645 acc 0.739 | valid loss 0.551 acc 0.806
Época  6 | train loss 0.638 acc 0.761 | valid loss 0.432 acc 0.866
Época  7 | train loss 0.581 acc 0.773 | valid loss 0.436 acc 0.826
Época  8 | train loss 0.532 acc 0.781 | valid loss 0.386 acc 0.881
Época  9 | train loss 0.532 acc 0.797 | valid loss 0.422 acc 0.858
Época 10 | train loss 0.531 acc 0.788 | valid loss 0.457 acc 0.810
Época 11 | train loss 0.467 acc 0.804 | valid loss 0.357 acc 0.885
Época 12 | train loss 0.413 acc 0.820 | valid loss 0.383 acc 0.866
Época 13 | train loss 0.404 acc 0.850 | valid loss 0.351 acc 0.881
Época 14 | train loss 0.364 acc 0.853 | valid loss 0.369 acc 0.870
Época 15 | train loss 0.448 acc 0.818 | valid loss 0.366 acc 0

In [18]:
torch.save(modelo.state_dict(), os.path.join(RUTA, "cnn_5s.pt"))

In [20]:
y_real, y_pred = [], []

modelo.eval()
with torch.no_grad():
    for imgs, etiquetas in dl_test:
        imgs = imgs.to(device)
        salidas = modelo(imgs)

        y_pred.extend(salidas.argmax(1).cpu().numpy())
        y_real.extend(etiquetas.numpy())

y_real = np.array(y_real)
y_pred = np.array(y_pred)
print(len(y_real), len(y_pred))

277 277


In [21]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

nombres = ["andar", "sentarse", "levantarse", "agacharse", "beber", "caerse"]

print("Precisión test:", accuracy_score(y_real, y_pred))
print(confusion_matrix(y_real, y_pred))
print(classification_report(y_real, y_pred, target_names=nombres, digits=3))

Precisión test: 0.9061371841155235
[[48  0  0  0  0  0]
 [ 4 44  0  0  0  0]
 [ 0  0 48  0  0  0]
 [ 1  0  1 42  2  3]
 [ 0  2  1 11 34  0]
 [ 1  0  0  0  0 35]]
              precision    recall  f1-score   support

       andar      0.889     1.000     0.941        48
    sentarse      0.957     0.917     0.936        48
  levantarse      0.960     1.000     0.980        48
   agacharse      0.792     0.857     0.824        49
       beber      0.944     0.708     0.810        48
      caerse      0.921     0.972     0.946        36

    accuracy                          0.906       277
   macro avg      0.911     0.909     0.906       277
weighted avg      0.910     0.906     0.904       277

